# Retrieval evaluation

Measure retrieval quality (hit rate and MRR) across different search configurations and pick the best. The flow is:

1. **Generate ground truth** — use an LLM to create realistic questions from sample papers, each labelled with its source document.
2. **Define metrics** — hit rate and MRR against those labels.
3. **Sweep configurations** — full-text `attributesToSearchOn`, ranking rules, top-k, and hybrid weights (with and without re-ranking).
4. **Pick the winner** — hybrid weights `[0.25, 0.75]` (keyword, vector) with re-ranking.

In [ ]:
import sys
sys.path.append("..")
import json
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

from tools import search, vector_search, fulltext_search, hybrid_search

load_dotenv()

## Generate ground-truth questions

Prompt an LLM to write realistic questions from each sample paper, keeping the source document as the relevance label.

In [2]:
data_gen_instructions = """
You are simulating a computer science student or researcher who is writing a research paper or trying to understand a technical topic.

Given the record below, generate 5 realistic questions that a student or researcher might ask about the topic.

Requirements:
- Each question must be answerable using information contained in the record.
- Questions should focus on the main topic or concepts discussed in the record, not on the record or paper itself.
- Questions should sound natural, like questions people would ask on Stack Overflow, Reddit, research discussions, or while studying.
- Questions should be clear and self-contained, but not overly formal.
- Avoid questions that are too broad (e.g. "What is concurrency?") or too narrow (e.g. asking for a specific sentence, figure, or minor detail).
- Prefer questions that require understanding or connecting concepts rather than simple keyword lookup.
- Use as few words from the original record as possible. Rephrase concepts in your own words where appropriate.
- Do not mention "this paper", "the record", "the author", or "the document".
- Do not assume information that is not present in the record.
- Keep each question reasonably concise: about 10–25 words.
- Make the 5 questions meaningfully different from each other.

Output only the 5 questions, numbered 1–5.
""".strip()

In [ ]:
class Questions(BaseModel):
    questions: list[str]

In [ ]:
def calc_price(usage):
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (usage["input_tokens"] / 1_000_000) * input_price_per_million
    output_cost = (usage["output_tokens"] / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

def calc_total_price(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_price(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost

In [ ]:
def generate_ground_truth_questions(doc, filename):
    llm = ChatOpenAI(model="gpt-5.4-mini")
    structured_llm = llm.with_structured_output(Questions, include_raw=True)
    result = structured_llm.invoke(
        [
            ("system", data_gen_instructions),
            ("user", doc["content"]),
        ]
    )
    usage = result["raw"].usage_metadata
    results = []
    questions = result["parsed"]
    for question in questions.questions:
        results.append(
            {
                "question": question,
                "title": doc["title"],
                "filename": filename,
            }
        )

    return results, usage

In [38]:
samples_dir = Path("samples")
docs = sorted(samples_dir.glob("*.json"))


json_file = docs[0]
with json_file.open("r", encoding="utf-8") as f:
    sample_data = json.load(f)
results, cost = generate_ground_truth_questions(sample_data, filename = json_file.name)
results


[{'question': '1. How does Ethereum decide the canonical chain after the move from proof of work to proof of stake?',
  'title': 'Ethereum: a secure decentralized generalized transaction ledger',
  'filename': 'digital_currency__paper.json'},
 {'question': '2. What makes an Ethereum transaction valid before execution, and how do gas limits and fees affect that?',
  'title': 'Ethereum: a secure decentralized generalized transaction ledger',
  'filename': 'digital_currency__paper.json'},
 {'question': '3. How do contract creation and normal message calls differ in their inputs, outputs, and state changes?',
  'title': 'Ethereum: a secure decentralized generalized transaction ledger',
  'filename': 'digital_currency__paper.json'},
 {'question': '4. Why are base fee and priority fee separated, and how do they influence transaction inclusion?',
  'title': 'Ethereum: a secure decentralized generalized transaction ledger',
  'filename': 'digital_currency__paper.json'},
 {'question': '5. How d

In [39]:
ground_truth = []
usages = []
for doc in tqdm(docs):
    with doc.open("r", encoding="utf-8") as f:
        sample_data = json.load(f)
    results, cost = generate_ground_truth_questions(sample_data, filename = doc.name)
    ground_truth.extend(results)
    usages.append(cost)

  0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
#df_ground_truth = pd.DataFrame(ground_truth)
#df_ground_truth.to_csv("ground_truth.csv", index=False)

## Load ground truth

Load the saved question set and run one search as a sanity check.

In [3]:
df_ground_truth = pd.read_csv("ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [32]:
results = search.invoke(ground_truth[0]["question"])
for result in results:
    print(result['title'])

Ethereum: a secure decentralized generalized transaction ledger
Ethereum: a secure decentralized generalized transaction ledger
Ethereum: a secure decentralized generalized transaction ledger
A History of the Virtual Synchrony Replication Model
Implementing Fault-Tolerant Services Using the State Machine Approach: A Tutorial
Statecharts: A Visual Formalism for Complex Systems
Optimistic replication
Optimistic replication
Statecharts: A Visual Formalism for Complex Systems
Out of the Tar Pit


## Retrieval metrics: hit rate & MRR

A result is relevant when its paper title matches the question's source. Hit rate = share of questions with at least one relevant hit; MRR rewards ranking the relevant hit higher.

In [7]:
def compute_relevance_text(q, search):
    question = q["question"]
    title = q["title"]
    results = search(question)
    relevant = []
    for result in results:
        relevant.append(int(result["title"] == title))
    return relevant

def compute_relevance_total(ground_truth, search):
    total_relevant = []
    for q in ground_truth:
        total_relevant.append(compute_relevance_text(q, search))
    return total_relevant

In [8]:
def hit_rate(relevant_total):
    total_hits = 0
    total_questions = len(relevant_total)
    for relevant in relevant_total:
        if 1 in relevant:
            total_hits += 1
    return total_hits / total_questions

In [9]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [10]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [30]:
evaluation_vector = evaluate(ground_truth, vector_search)


## Full-text: `attributesToSearchOn` evaluation

In [34]:
configs = [
    ["title", "tags", "content"],
    ["tags", "title", "content"],
    ["content", "title", "tags"],
    ["content", "title"],
    ["title", "content"],
    ["content", "tags"],
    ["content"],
    ["title", "tags"],
    ["title"],
]

rows = []
for config in configs:
    result = evaluate(ground_truth,
                      lambda query, num_results=5, c=config: fulltext_search(query, num_results, attributesToSearchOn=c))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)


,config,hit_rate,mrr
0,"['title', 'tags', 'content']",0.709091,0.555455
1,"['tags', 'title', 'content']",0.709091,0.555455
2,"['content', 'title', 'tags']",0.709091,0.555455
3,"['content', 'title']",0.709091,0.555455
4,"['title', 'content']",0.709091,0.555455
5,"['content', 'tags']",0.654545,0.541515
6,['content'],0.654545,0.541515
7,"['title', 'tags']",0.072727,0.072727
8,['title'],0.072727,0.072727


`tags` doesn't seem to have any impact on the search results, and the order of `attributesToSearchOn` doesn't matter either. The only thing that matters is whether `content` is included: with it the results are good, without it they are bad. `title` has a small effect. With `content` only, hit rate is around 0.65 and MRR around 0.54; adding `title` boosts hit rate to 0.71 and MRR to 0.55 (a small gain).

## Ranking rule evaluation

In [42]:
configs = [
    ["attribute", "words", "proximity", "typo", "sort"],
    ["words", "proximity", "typo", "attribute", "sort"],
    ["words", "typo", "proximity", "attribute", "sort"],
    ["typo", "words", "proximity", "attribute", "sort"],
    ["proximity", "typo", "words", "attribute", "sort"],
    ["sort", "typo", "words", "attribute", "proximity"],
]

rows = []
for config in configs:
    result = evaluate(ground_truth,
                      lambda query, num_results=5, c=config: fulltext_search(query, num_results, rankingRules=c))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)



,config,hit_rate,mrr
0,"['attribute', 'words', 'proximity', 'typo', 's...",0.654545,0.543636
1,"['words', 'proximity', 'typo', 'attribute', 's...",0.709091,0.555455
2,"['words', 'typo', 'proximity', 'attribute', 's...",0.709091,0.555455
3,"['typo', 'words', 'proximity', 'attribute', 's...",0.709091,0.555455
4,"['proximity', 'typo', 'words', 'attribute', 's...",0.709091,0.555455
5,"['sort', 'typo', 'words', 'attribute', 'proxim...",0.709091,0.555455


Only `["attribute", "words", "proximity", "typo", "sort"]` has the lowest scores among the 6 ranking rules. The other 5 ranking rules have the same score. The order of the ranking rules doesn't seem to matter as much as the presence of the ranking rules.

## Top K evaluation fulltext search

In [36]:
k = [5, 10, 20, 50, 100]
rows = []
for config in k:
    result = evaluate(ground_truth,
                      lambda query, num_results=config: fulltext_search(query, num_results))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)

,config,hit_rate,mrr
0,5,0.709091,0.555455
1,10,0.763636,0.562468
2,20,0.854545,0.568422
3,50,0.854545,0.568422
4,100,0.872727,0.568633


When k is 100, it has the highest hit rate but the hit rate is expected as the more the results return the more likely the relevant doc will be included. The MRR doesn't change much across all the different k values. Top k = 20 has more balanced hit rate and MRR. This is also much appropriate for RAG as the number of relevant docs is usually small and top k = 20 is a good balance between hit rate and MRR. 

## Top K evaluation vector search

In [37]:
k = [5, 10, 20, 50, 100]
rows = []
for config in k:
    result = evaluate(ground_truth,
                      lambda query, num_results=config: vector_search(query, num_results))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)

,config,hit_rate,mrr
0,5,1.0,0.855152
1,10,1.0,0.855152
2,20,1.0,0.855152
3,50,1.0,0.855152
4,100,1.0,0.855152


VectorSearch scores are the same among different k values. This suggest that top k = 5 is already enough to get the relevant docs.

## Hybrid search weights (without re-ranking)

In [44]:
configs = [[1.0, 1.0], [0.5, 0.5], [0.25, 0.75], [0.75, 0.25], [1.0, 0.0], [0.0, 1.0]]
rows = []
for config in configs:
    result = evaluate(ground_truth,
                      lambda query, num_results=5, weights=config: hybrid_search(query, num_results, weights, rerank=False))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)

,config,hit_rate,mrr
0,"[1.0, 1.0]",0.945455,0.716364
1,"[0.5, 0.5]",0.945455,0.716364
2,"[0.25, 0.75]",1.000000,0.846667
3,"[0.75, 0.25]",0.745455,0.575455
4,"[1.0, 0.0]",0.690909,0.542121
5,"[0.0, 1.0]",1.000000,0.835455


`0.25, 0.75` (fulltext, vector) has the best hit rate and MRR. This suggests that vector search contributes more to the relevance of the results than fulltext search. But fulltext search boosts MRR score only around 0.01 which is very minor. This can be seen in the results of `0.0, 1.0` (fulltext, vector) which has almost the same hit rate and MRR as `0.25, 0.75` (fulltext, vector). 

## Hybrid search weights (with re-ranking)

In [39]:
configs = [[1.0, 1.0], [0.5, 0.5], [0.25, 0.75], [0.75, 0.25], [1.0, 0.0], [0.0, 1.0]]
rows = []
for config in configs:
    result = evaluate(ground_truth,
                      lambda query, num_results=5, weights=config: hybrid_search(query, num_results, weights))
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)

,config,hit_rate,mrr
0,"[1.0, 1.0]",0.945455,0.909091
1,"[0.5, 0.5]",0.945455,0.909091
2,"[0.25, 0.75]",1.000000,0.923333
3,"[0.75, 0.25]",0.745455,0.727273
4,"[1.0, 0.0]",0.690909,0.676364
5,"[0.0, 1.0]",1.000000,0.915758


The observation is the same weight `[0.25, 0.75]` (fulltext, vector) has the best hit rate and MRR with re-ranking as well. But with re-ranking, the MRR score is boosted by around 0.08 which is significant. This suggests that re-ranking is very important for hybrid search.

## Hybrid search top-k (with re-ranking)

In [45]:
k = [5, 10, 20, 50, 100]
rows = []
for config in k:
    result = evaluate(ground_truth,
                      lambda query, num_results=config: search.invoke({"query": query, "num_results": num_results}))
    print(result)
    rows.append({"config": str(config), "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

pd.DataFrame(rows)


{'hit_rate': 1.0, 'mrr': 0.9233333333333333}
{'hit_rate': 1.0, 'mrr': 0.9460606060606062}
{'hit_rate': 1.0, 'mrr': 0.9393939393939394}
{'hit_rate': 1.0, 'mrr': 0.9363636363636364}
{'hit_rate': 1.0, 'mrr': 0.9454545454545454}


,config,hit_rate,mrr
0,5,1.0,0.923333
1,10,1.0,0.946061
2,20,1.0,0.939394
3,50,1.0,0.936364
4,100,1.0,0.945455


There is an improvement in the scores as k grows, but it is not significant — hit rate and MRR are already good at k = 5. So top-k = 5 is enough for hybrid search too.